# Whole-paper reproduction status

This notebook is the control surface for reproducing the manuscript from publication-safe derived inputs. It inventories the numerical products, points to the analysis notebooks, and keeps unresolved manuscript/code choices visible. It does not substitute a preferred analysis for an unresolved one.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
ROOT

## Registered numerical products

A missing product here is a pipeline failure or an analysis that has not yet been integrated into the public run.

In [ ]:
products = {
    'DSWP labels': ROOT / 'results/dswp/buzz_events_labelled.csv',
    'DSWP mixed depth model': ROOT / 'results/dswp/depth_mixed_model.json',
    'DSWP trajectory energetics': ROOT / 'results/dswp/individual_energy.csv',
    'DSWP Aoki comparison': ROOT / 'results/dswp/aoki_individual_energy.csv',
    'DSWP Markov transitions': ROOT / 'results/dswp/buzz_markov_transitions.csv',
    'DSWP recurrence, manuscript labels': ROOT / 'results/dswp/comeback_association_legacy_manuscript.json',
    'DSWP recurrence, current labels': ROOT / 'results/dswp/comeback_association_current_classifier.json',
    'Trajectory population summary': ROOT / 'results/trajectory/population_summary.json',
    'Azores summary': ROOT / 'results/azores/azores_summary.json',
    'CETI 2024 kinematic statistics': ROOT / 'results/ceti2024/kinematic_statistics.csv',
}
inventory = pd.DataFrame([{'product': name, 'path': str(path.relative_to(ROOT)), 'available': path.exists()} for name, path in products.items()])
inventory

## Notebook map

Each row identifies the notebook responsible for explaining and regenerating a manuscript analysis. Figure notebooks may compose already-tested result tables; the analytical modules under `src/` remain authoritative.

In [ ]:
notebook_rows = [
    ('Cohort, buzz construction, threshold', 'notebooks/dswp/00_dataset_and_buzz_construction.ipynb'),
    ('DSWP event-level analyses', 'notebooks/dswp/01_event_level_analysis.ipynb'),
    ('Energetic-model derivation', 'notebooks/dswp/02_energy_model_derivation.ipynb'),
    ('DSWP manuscript components', 'notebooks/dswp/03_manuscript_figure_components.ipynb'),
    ('Spatial recurrence alternatives', 'notebooks/dswp/04_comeback_reproduction_audit.ipynb'),
    ('Trajectory-scale registered analysis', 'notebooks/trajectory/01_reconstructed_trajectory_analysis.ipynb'),
    ('Trajectory manuscript and SI figures', 'notebooks/trajectory/02_manuscript_trajectory_figures.ipynb'),
    ('Azores source provenance', 'notebooks/validation/01_azores_open_data.ipynb'),
    ('CETI 2024 derived validation', 'notebooks/validation/02_ceti2024_derived_validation.ipynb'),
    ('Azores numerical derivation', 'notebooks/validation/03_azores_derivation.ipynb'),
    ('Azores manuscript figures', 'notebooks/validation/04_azores_manuscript_figures.ipynb'),
    ('CETI 2024 manuscript figures', 'notebooks/validation/05_ceti2024_manuscript_figures.ipynb'),
]
notebook_map = pd.DataFrame(notebook_rows, columns=['scope', 'notebook'])
notebook_map['available'] = notebook_map['notebook'].map(lambda value: (ROOT / value).exists())
notebook_map

## Known issue touchpoints

Known issue touchpoint: the following choices can change manuscript numbers or captions. They must remain visible in the corresponding notebooks until the manuscript and production implementation agree:

- plateau-threshold null optimum and the raw-audio/full-record boundary;
- reconstruction-error aggregation and location-bearing map boundary;
- classifier speed variable and the diagnostic duration definition;
- recurrence thresholds and legacy versus registered classifier labels;
- DSWP Markov resampling unit and handling of ambiguous gaps;
- terminal-buzz count and pause-analysis denominator;
- Azores operative 29/28/1 labels versus balanced 29/29 sensitivity, echo window, and corrected bootstrap;
- the custom Levy-like fitter named in Methods;
- the buzz-centred MSD time-origin bug and archived/current classifier split; and
- the unavailable CETI 2024 roll-speed time course.

The detailed evidence and required manuscript decisions are in `docs/known_issues.md`.

In [ ]:
headline = {}
if products['Trajectory population summary'].exists():
    headline['trajectory'] = json.loads(products['Trajectory population summary'].read_text())
if products['Azores summary'].exists():
    headline['azores'] = json.loads(products['Azores summary'].read_text())
if products['DSWP trajectory energetics'].exists():
    energy = pd.read_csv(products['DSWP trajectory energetics'])
    headline['energy_mean_MJ'] = energy[['A', 'B']].mean().div(1e6).to_dict()
headline

## Reproduction boundary

This repository targets result reproduction from reviewed derived data. Raw DTAG audio and sensor streams, original identifiers, GPS coordinates, and private crosswalks are intentionally excluded. Panels requiring those inputs must use a steward-approved derived export or remain explicitly non-reproducible from the public release.